# 🦷 Dental Index Prediction — DINOv2 Training Notebook

**Architecture:** DINOv2-base (ViT-B/14, 86M params) frozen backbone + lightweight MLP classifier head

**Strategy:**
1. Extract features once using frozen DINOv2 backbone (no backbone training)
2. Train only a small MLP head (~300K params) on cached features
3. 5-fold stratified cross-validation with ensemble averaging
4. Weighted cross-entropy loss for class imbalance

Predicts 3 dental indices from 3 photograph views:
- **MGI** (Modified Gingival Index): 0–4
- **OHI** (Oral Hygiene Index): 0–3  
- **GEI** (Gingival Enlargement Index): 0–2

**Why DINOv2?**
- **DINOv2-base** (ViT-B/14, 86M params) was self-supervised on **142M diverse images**
- Produces rich visual features that transfer well even with very few samples (201)
- Frozen backbone = zero risk of overfitting the backbone

## 1. Setup & Configuration

In [ ]:
import os, sys, gc, json, time, random, warnings
import numpy as np
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, WeightedRandomSampler, TensorDataset

import timm
import matplotlib.pyplot as plt
import matplotlib
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# CPU optimization
if not torch.cuda.is_available():
    _n_threads = os.cpu_count() or 4
    torch.set_num_threads(_n_threads)
    try:
        torch.set_num_interop_threads(max(1, _n_threads // 2))
    except RuntimeError:
        pass

# GPU: enable TF32 for Ampere+ GPUs and cuDNN benchmarking
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

# Resolve project root
try:
    PROJECT_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd()
    if (PROJECT_ROOT / 'Train_Model.ipynb').exists() and not (PROJECT_ROOT / 'Thesis_Data').exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'Thesis_Data').exists():
    PROJECT_ROOT = Path(r'E:\Dental_Project')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.dataset import DentalDataset
from ml.model import DINOv2MultiViewModel, DentalClassifierHead
from ml.transforms import get_val_transforms

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['figure.dpi'] = 110
warnings.filterwarnings('ignore', category=UserWarning)

# AMP (automatic mixed precision) — use float16 on GPU for 2-3x speedup
USE_AMP = torch.cuda.is_available()

print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB)')
    print(f'AMP (float16): {USE_AMP}  |  TF32: {torch.backends.cuda.matmul.allow_tf32}')
    print(f'cuDNN benchmark: {torch.backends.cudnn.benchmark}')
else:
    print(f'Training on CPU  |  Threads: {torch.get_num_threads()}')
    print(f'AMP: disabled (CPU)')
print(f'timm: {timm.__version__}')
print(f'Project root: {PROJECT_ROOT}')
assert (PROJECT_ROOT / 'Thesis_Data').exists(), f'Thesis_Data not found at {PROJECT_ROOT}'

In [ ]:
CONFIG = {
    'data_dir':         str(PROJECT_ROOT / 'Thesis_Data'),
    'output_dir':       str(PROJECT_ROOT / 'ml' / 'checkpoints'),
    'image_size':       224,
    'seed':             42,
    'n_folds':          5,
    'epochs_per_fold':  60,
    'batch_size':       64,       # training on cached features, so can be large
    'lr':               1e-3,
    'weight_decay':     1e-3,
    'patience':         15,
    'dropout':          0.4,
    'label_smoothing':  0.1,
    'backbone':         'vit_base_patch14_reg4_dinov2',
    'feature_dim':      768,      # per-view
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Reproducibility
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:20s}: {v}')
print(f'\nDevice: {device}')

In [ ]:
# Load full dataset (no augmentation — we just need raw images for feature extraction)
transform = get_val_transforms(CONFIG['image_size'])
full_dataset = DentalDataset(CONFIG['data_dir'], transform=transform)
print(f'Total samples: {len(full_dataset)}')

# Class distributions
mgi_dist = np.bincount([s['mgi'] for s in full_dataset.samples], minlength=5)
ohi_dist = np.bincount([s['ohi'] for s in full_dataset.samples], minlength=4)
gei_dist = np.bincount([s['gei'] for s in full_dataset.samples], minlength=3)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, dist, title, colors in zip(axes,
    [mgi_dist, ohi_dist, gei_dist],
    ['MGI (0-4)', 'OHI (0-3)', 'GEI (0-2)'],
    [['#77dd77','#b5e61d','#fdfd96','#ff6961','#ff1a1a'],
     ['#77dd77','#b5e61d','#daa520','#ff6961'],
     ['#77dd77','#daa520','#ff6961']]):
    bars = ax.bar(range(len(dist)), dist, color=colors[:len(dist)])
    ax.set_title(title, fontweight='bold')
    for bar, v in zip(bars, dist):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'MGI distribution: {dict(enumerate(mgi_dist.tolist()))}')
print(f'OHI distribution: {dict(enumerate(ohi_dist.tolist()))}')
print(f'GEI distribution: {dict(enumerate(gei_dist.tolist()))}')

## 2. DINOv2 Feature Extraction

In [ ]:
# Load DINOv2-base backbone (frozen — no training, just feature extraction)
feature_cache_path = os.path.join(CONFIG['output_dir'], 'dinov2_features.npz')

def _interpolate_pos_embed(checkpoint_pos_embed, model_pos_embed, patch_size=14):
    """Interpolate pos_embed from pretrained resolution to model's resolution."""
    import torch.nn.functional as Fn
    tgt_len = model_pos_embed.shape[1]
    src_len = checkpoint_pos_embed.shape[1]
    dim = checkpoint_pos_embed.shape[2]
    if src_len == tgt_len:
        return checkpoint_pos_embed

    # DINOv2 checkpoint pos_embed = [1, 1+num_patches, dim] (1 for CLS)
    # timm with no_embed_class=True expects [1, num_patches, dim] (no CLS)
    cls_token = checkpoint_pos_embed[:, :1, :]
    patch_pos = checkpoint_pos_embed[:, 1:, :]
    src_npatch = patch_pos.shape[1]

    if src_npatch == tgt_len:
        return patch_pos

    # Interpolate patch positions
    src_size = int(src_npatch ** 0.5)
    tgt_size = int(tgt_len ** 0.5)
    assert src_size * src_size == src_npatch, f'Source not square: {src_npatch}'
    patch_pos = patch_pos.reshape(1, src_size, src_size, dim).permute(0, 3, 1, 2).float()
    patch_pos = Fn.interpolate(patch_pos, size=(tgt_size, tgt_size), mode='bicubic', align_corners=False)
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, tgt_size * tgt_size, dim)
    return patch_pos

def _load_dinov2_backbone(model_name, device='cpu', img_size=224):
    """Load DINOv2 backbone with key/shape fixes for timm 1.0.25."""
    model = timm.create_model(model_name, pretrained=False, num_classes=0,
                               global_pool='avg', img_size=img_size)
    cfg = model.pretrained_cfg
    url = cfg.get('url', '')
    print(f'  Downloading weights from: {url}')
    sd = torch.hub.load_state_dict_from_url(url, map_location=device, progress=True)

    model_sd = model.state_dict()
    remapped = {}
    for k, v in sd.items():
        target_key = k
        if k == 'norm.weight':
            target_key = 'fc_norm.weight'
        elif k == 'norm.bias':
            target_key = 'fc_norm.bias'
        elif k == 'mask_token':
            continue

        if target_key not in model_sd:
            continue

        if 'pos_embed' in target_key and v.shape != model_sd[target_key].shape:
            v = _interpolate_pos_embed(v, model_sd[target_key])
            print(f'  Interpolated pos_embed: {sd[k].shape} \u2192 {v.shape}')

        if v.shape != model_sd[target_key].shape:
            print(f'  Skipping {target_key}: {v.shape} vs {model_sd[target_key].shape}')
            continue

        remapped[target_key] = v

    result = model.load_state_dict(remapped, strict=False)
    missing = [k for k in result.missing_keys if 'head' not in k]
    if missing:
        print(f'  Note: missing keys: {missing}')
    print(f'  Loaded {len(remapped)} / {len(model_sd)} parameters')
    return model.to(device)

if os.path.exists(feature_cache_path):
    print('Loading cached DINOv2 features...')
    cached = np.load(feature_cache_path)
    all_features = cached['features']
    all_mgi = cached['mgi']
    all_ohi = cached['ohi']
    all_gei = cached['gei']
    all_ids = cached['ids']
    print(f'Loaded {len(all_features)} cached feature vectors ({all_features.shape[1]}-dim)')
else:
    print(f'Extracting DINOv2 features from {len(full_dataset)} samples...')
    print(f'Backbone: {CONFIG["backbone"]} (FROZEN, {CONFIG["feature_dim"]}-dim per view)')
    print(f'Device: {device} | AMP: {USE_AMP}')

    backbone = _load_dinov2_backbone(CONFIG['backbone'], device, CONFIG['image_size'])
    backbone.eval()

    # Batch size for feature extraction — larger on GPU
    FE_BATCH = 16 if torch.cuda.is_available() else 4

    # Pre-load and stack all images by view for batch processing
    print(f'Loading images (batch size = {FE_BATCH})...')
    frontal_imgs, left_imgs, right_imgs = [], [], []
    all_mgi, all_ohi, all_gei, all_ids = [], [], [], []

    for i in range(len(full_dataset)):
        sample = full_dataset[i]
        frontal_imgs.append(sample['frontal'])
        left_imgs.append(sample['left_lateral'])
        right_imgs.append(sample['right_lateral'])
        all_mgi.append(sample['mgi_label'])
        all_ohi.append(sample['ohi_label'])
        all_gei.append(sample['gei_label'])
        all_ids.append(full_dataset.samples[i]['sl_no'])

    frontal_imgs = torch.stack(frontal_imgs)
    left_imgs = torch.stack(left_imgs)
    right_imgs = torch.stack(right_imgs)

    n_samples = len(frontal_imgs)
    all_features = []

    t0 = time.time()
    with torch.no_grad():
        for start in tqdm(range(0, n_samples, FE_BATCH), desc='Extracting features'):
            end = min(start + FE_BATCH, n_samples)
            batch_f = frontal_imgs[start:end].to(device)
            batch_l = left_imgs[start:end].to(device)
            batch_r = right_imgs[start:end].to(device)

            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    feat_f = backbone(batch_f)
                    feat_l = backbone(batch_l)
                    feat_r = backbone(batch_r)
            else:
                feat_f = backbone(batch_f)
                feat_l = backbone(batch_l)
                feat_r = backbone(batch_r)

            feat = torch.cat([feat_f, feat_l, feat_r], dim=1).cpu().numpy()
            all_features.append(feat)

            # Free GPU memory per batch
            del batch_f, batch_l, batch_r, feat_f, feat_l, feat_r
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    elapsed = time.time() - t0
    all_features = np.concatenate(all_features, axis=0)
    all_mgi = np.array(all_mgi)
    all_ohi = np.array(all_ohi)
    all_gei = np.array(all_gei)
    all_ids = np.array(all_ids)

    # Free image tensors
    del frontal_imgs, left_imgs, right_imgs
    gc.collect()

    # Cache to disk so we don't need to re-extract
    np.savez(feature_cache_path, features=all_features, mgi=all_mgi,
             ohi=all_ohi, gei=all_gei, ids=all_ids)

    del backbone
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f'Extracted and cached {len(all_features)} feature vectors in {elapsed:.0f}s')

print(f'Feature shape: {all_features.shape}  (samples \u00d7 3\u00d7{CONFIG["feature_dim"]})')
print(f'Labels: MGI {np.bincount(all_mgi)}, OHI {np.bincount(all_ohi)}, GEI {np.bincount(all_gei)}')

## 3. Class Weights & Loss Setup

In [ ]:
# Compute class weights for imbalanced data
def compute_class_weights(labels, n_classes):
    """Inverse frequency weighting, normalized."""
    counts = np.bincount(labels, minlength=n_classes).astype(float) + 1e-6
    weights = 1.0 / counts
    weights = weights / weights.sum() * n_classes
    return torch.FloatTensor(weights).to(device)

mgi_weights = compute_class_weights(all_mgi, 5)
ohi_weights = compute_class_weights(all_ohi, 4)
gei_weights = compute_class_weights(all_gei, 3)

print(f'MGI weights: {mgi_weights.cpu().numpy().round(3)}')
print(f'OHI weights: {ohi_weights.cpu().numpy().round(3)}')
print(f'GEI weights: {gei_weights.cpu().numpy().round(3)}')

# Multi-task loss with class weights
class MultiTaskCELoss(nn.Module):
    """Weighted cross-entropy for 3 tasks with label smoothing."""
    def __init__(self, mgi_w, ohi_w, gei_w, smoothing=0.1):
        super().__init__()
        self.mgi_loss = nn.CrossEntropyLoss(weight=mgi_w, label_smoothing=smoothing)
        self.ohi_loss = nn.CrossEntropyLoss(weight=ohi_w, label_smoothing=smoothing)
        self.gei_loss = nn.CrossEntropyLoss(weight=gei_w, label_smoothing=smoothing)

    def forward(self, outputs, mgi_labels, ohi_labels, gei_labels):
        l_mgi = self.mgi_loss(outputs['mgi'], mgi_labels)
        l_ohi = self.ohi_loss(outputs['ohi'], ohi_labels)
        l_gei = self.gei_loss(outputs['gei'], gei_labels)
        return l_mgi + l_ohi + l_gei

criterion = MultiTaskCELoss(mgi_weights, ohi_weights, gei_weights, CONFIG['label_smoothing'])
print(f'\nLoss: MultiTaskCELoss (label_smoothing={CONFIG["label_smoothing"]}) \u2713')

## 4. K-Fold Cross-Validation Training

In [ ]:
def train_one_fold(fold_idx, train_idx, val_idx, features, mgi, ohi, gei):
    """Train a single fold's classifier head on cached features."""
    # Prepare tensors
    X_train = torch.FloatTensor(features[train_idx]).to(device)
    X_val = torch.FloatTensor(features[val_idx]).to(device)
    y_mgi_train = torch.LongTensor(mgi[train_idx]).to(device)
    y_ohi_train = torch.LongTensor(ohi[train_idx]).to(device)
    y_gei_train = torch.LongTensor(gei[train_idx]).to(device)
    y_mgi_val = torch.LongTensor(mgi[val_idx]).to(device)
    y_ohi_val = torch.LongTensor(ohi[val_idx]).to(device)
    y_gei_val = torch.LongTensor(gei[val_idx]).to(device)

    # Create DataLoaders
    train_ds = TensorDataset(X_train, y_mgi_train, y_ohi_train, y_gei_train)
    val_ds = TensorDataset(X_val, y_mgi_val, y_ohi_val, y_gei_val)
    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)

    # Build classifier head
    fused_dim = CONFIG['feature_dim'] * 3
    head = DentalClassifierHead(input_dim=fused_dim, dropout=CONFIG['dropout']).to(device)
    
    optimizer = torch.optim.AdamW(head.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs_per_fold'], eta_min=1e-6)

    # AMP scaler for GPU mixed precision training
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(1, CONFIG['epochs_per_fold'] + 1):
        # --- Train ---
        head.train()
        train_loss_sum, train_n = 0.0, 0
        for batch_feats, batch_mgi, batch_ohi, batch_gei in train_loader:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=USE_AMP):
                outputs = head(batch_feats)
                loss = criterion(outputs, batch_mgi, batch_ohi, batch_gei)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss_sum += loss.item() * len(batch_feats)
            train_n += len(batch_feats)
        train_loss = train_loss_sum / train_n

        # --- Validate ---
        head.eval()
        val_loss_sum, val_n = 0.0, 0
        val_preds = {'mgi': [], 'ohi': [], 'gei': []}
        val_trues = {'mgi': [], 'ohi': [], 'gei': []}
        with torch.no_grad():
            for batch_feats, batch_mgi, batch_ohi, batch_gei in val_loader:
                with torch.amp.autocast('cuda', enabled=USE_AMP):
                    outputs = head(batch_feats)
                    loss = criterion(outputs, batch_mgi, batch_ohi, batch_gei)
                val_loss_sum += loss.item() * len(batch_feats)
                val_n += len(batch_feats)
                for key, labels in [('mgi', batch_mgi), ('ohi', batch_ohi), ('gei', batch_gei)]:
                    val_preds[key].extend(outputs[key].argmax(1).cpu().tolist())
                    val_trues[key].extend(labels.cpu().tolist())
        val_loss = val_loss_sum / val_n

        accs = {k: accuracy_score(val_trues[k], val_preds[k]) for k in ['mgi', 'ohi', 'gei']}
        avg_acc = np.mean(list(accs.values()))
        scheduler.step()

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                        'accs': accs, 'avg_acc': avg_acc})

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in head.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1 or patience_counter == 0:
            star = ' \u2605' if patience_counter == 0 else ''
            tqdm.write(f'  Fold {fold_idx} Ep {epoch:3d} | TrLoss {train_loss:.4f} | '
                       f'VlLoss {val_loss:.4f} | MGI {accs["mgi"]:.3f} OHI {accs["ohi"]:.3f} '
                       f'GEI {accs["gei"]:.3f} | Avg {avg_acc:.3f}{star}')

        if patience_counter >= CONFIG['patience']:
            tqdm.write(f'  Fold {fold_idx}: Early stopping at epoch {epoch}')
            break

    # Load best weights
    head.load_state_dict(best_state)
    head.eval()

    # Final validation metrics
    with torch.no_grad():
        all_out = head(X_val)
    final_preds = {k: all_out[k].argmax(1).cpu().numpy() for k in ['mgi', 'ohi', 'gei']}
    final_trues = {'mgi': mgi[val_idx], 'ohi': ohi[val_idx], 'gei': gei[val_idx]}
    final_accs = {k: accuracy_score(final_trues[k], final_preds[k]) for k in ['mgi', 'ohi', 'gei']}

    return head, best_val_loss, final_accs, final_preds, final_trues, history

print('Training function ready \u2713')
print(f'AMP (mixed precision): {"enabled" if USE_AMP else "disabled"}')

### Run 5-Fold Cross-Validation
Each fold trains a classifier head on cached DINOv2 features.
Training is extremely fast since features are pre-extracted (no backbone computation).

In [ ]:
skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['seed'])

fold_heads = []
fold_val_losses = []
fold_accs = []
all_val_preds = {'mgi': np.zeros(len(all_features), dtype=int),
                 'ohi': np.zeros(len(all_features), dtype=int),
                 'gei': np.zeros(len(all_features), dtype=int)}
all_val_trues = {'mgi': all_mgi.copy(), 'ohi': all_ohi.copy(), 'gei': all_gei.copy()}
all_histories = []

print(f'Starting {CONFIG["n_folds"]}-fold cross-validation')
print(f'Epochs/fold: {CONFIG["epochs_per_fold"]} | Patience: {CONFIG["patience"]}')
print(f'Classifier head params: ~{sum(p.numel() for p in DentalClassifierHead(CONFIG["feature_dim"]*3).parameters()):,}')
print('='*70)

t0_total = time.time()

for fold, (train_idx, val_idx) in enumerate(skf.split(all_features, all_mgi), 1):
    print(f'\n--- Fold {fold}/{CONFIG["n_folds"]} (train={len(train_idx)}, val={len(val_idx)}) ---')
    t0 = time.time()

    head, val_loss, accs, preds, trues, history = train_one_fold(
        fold, train_idx, val_idx, all_features, all_mgi, all_ohi, all_gei)

    elapsed = time.time() - t0
    fold_heads.append(head)
    fold_val_losses.append(val_loss)
    fold_accs.append(accs)
    all_histories.append(history)

    # Store OOF predictions
    for key in ['mgi', 'ohi', 'gei']:
        all_val_preds[key][val_idx] = preds[key]

    # Save fold head
    fold_path = os.path.join(CONFIG['output_dir'], f'fold_{fold}_head.pth')
    torch.save({'head_state_dict': head.state_dict(), 'fold': fold,
                'val_loss': val_loss, 'accs': accs, 'config': CONFIG}, fold_path)

    print(f'  Fold {fold} done in {elapsed:.1f}s | ValLoss={val_loss:.4f} | '
          f'MGI={accs["mgi"]:.3f} OHI={accs["ohi"]:.3f} GEI={accs["gei"]:.3f} | '
          f'Avg={np.mean(list(accs.values())):.3f}')

total_time = time.time() - t0_total
print(f'\n{"="*70}')
print(f'All {CONFIG["n_folds"]} folds complete in {total_time:.1f}s')
mean_accs = {k: np.mean([f[k] for f in fold_accs]) for k in ['mgi', 'ohi', 'gei']}
print(f'Mean Accuracy  \u2014 MGI: {mean_accs["mgi"]:.3f}  OHI: {mean_accs["ohi"]:.3f}  GEI: {mean_accs["gei"]:.3f}')
print(f'Mean Overall   \u2014 {np.mean(list(mean_accs.values())):.3f}')

# Save a single best_model.pth for Django compatibility
best_fold = np.argmin(fold_val_losses) + 1
import shutil
shutil.copy(os.path.join(CONFIG['output_dir'], f'fold_{best_fold}_head.pth'),
            os.path.join(CONFIG['output_dir'], 'best_model.pth'))
print(f'\nBest fold: {best_fold} (ValLoss={fold_val_losses[best_fold-1]:.4f})')
print(f'Saved best_model.pth + all fold_*_head.pth for ensemble inference')

## 5. Out-of-Fold Evaluation

In [ ]:
# Out-of-fold predictions: every sample was predicted exactly once (when it was in val set)
print('Out-of-Fold Classification Reports (aggregated across all folds):')
print('='*70)

for key, n_cls, name in [('mgi', 5, 'MGI'), ('ohi', 4, 'OHI'), ('gei', 3, 'GEI')]:
    print(f'\n--- {name} ---')
    print(classification_report(all_val_trues[key], all_val_preds[key],
                                labels=list(range(n_cls)), zero_division=0))

# Overall accuracy
oof_accs = {k: accuracy_score(all_val_trues[k], all_val_preds[k]) for k in ['mgi', 'ohi', 'gei']}
print(f'\nOOF Accuracy \u2014 MGI: {oof_accs["mgi"]:.3f}  OHI: {oof_accs["ohi"]:.3f}  GEI: {oof_accs["gei"]:.3f}')
print(f'OOF Average  \u2014 {np.mean(list(oof_accs.values())):.3f}')

## 6. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, key, n_cls, name in zip(axes,
    ['mgi', 'ohi', 'gei'], [5, 4, 3], ['MGI', 'OHI', 'GEI']):
    cm = confusion_matrix(all_val_trues[key], all_val_preds[key], labels=list(range(n_cls)))
    im = ax.imshow(cm, cmap='Blues', interpolation='nearest')
    ax.set_title(f'{name} Confusion Matrix\n(OOF Acc: {oof_accs[key]:.1%})', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks(range(n_cls))
    ax.set_yticks(range(n_cls))
    for i in range(n_cls):
        for j in range(n_cls):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'confusion_matrices_oof.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Fold-wise Accuracy Summary

In [ ]:
# --- Plot per-fold accuracy bars ---
fold_nums = list(range(1, CONFIG['n_folds'] + 1))
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
bar_w = 0.6
for ax, key, name in zip(axes, ['mgi', 'ohi', 'gei'], ['MGI', 'OHI', 'GEI']):
    accs_list = [fold_accs[f][key] for f in range(CONFIG['n_folds'])]
    bars = ax.bar(fold_nums, accs_list, width=bar_w, color='steelblue', edgecolor='navy', alpha=0.85)
    ax.axhline(np.mean(accs_list), color='red', ls='--', lw=1.5, label=f'Mean: {np.mean(accs_list):.1%}')
    ax.set_title(f'{name} Fold Accuracy', fontweight='bold')
    ax.set_xlabel('Fold')
    ax.set_ylabel('Accuracy')
    ax.set_ylim(0, 1)
    ax.set_xticks(fold_nums)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    for bar, acc in zip(bars, accs_list):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{acc:.1%}', ha='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'fold_accuracies.png'), dpi=150, bbox_inches='tight')
plt.show()

# --- Per-fold training curves ---
fig, axes = plt.subplots(1, CONFIG['n_folds'], figsize=(4 * CONFIG['n_folds'], 4), squeeze=False)
for fold_idx in range(CONFIG['n_folds']):
    ax = axes[0, fold_idx]
    hist = all_histories[fold_idx]
    epochs = [h['epoch'] for h in hist]
    ax.plot(epochs, [h['train_loss'] for h in hist], 'b-', lw=1.5, label='Train')
    ax.plot(epochs, [h['val_loss'] for h in hist], 'r-', lw=1.5, label='Val')
    ax.set_title(f'Fold {fold_idx+1} Loss', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Ensemble Model for Django

In [ ]:
# Save ensemble info checkpoint for Django
ensemble_ckpt = {
    'n_folds': CONFIG['n_folds'],
    'backbone': CONFIG['backbone'],
    'feature_dim': CONFIG['feature_dim'],
    'fold_accs': {f: fold_accs[f] for f in range(CONFIG['n_folds'])},
    'oof_accs': oof_accs,
}
torch.save(ensemble_ckpt, os.path.join(CONFIG['output_dir'], 'ensemble_info.pth'))

# Save training history
history_data = {
    'fold_accs': fold_accs,
    'oof_accs': oof_accs,
    'config': CONFIG,
}
with open(os.path.join(CONFIG['output_dir'], 'training_history.json'), 'w') as fp:
    json.dump(history_data, fp, indent=2, default=str)

print(f'Saved: best_model.pth, ensemble_info.pth, training_history.json')
print(f'Fold head files: fold_1_head.pth .. fold_{CONFIG["n_folds"]}_head.pth')
print(f'\nAll files in {CONFIG["output_dir"]}:')
for fn in sorted(os.listdir(CONFIG['output_dir'])):
    sz = os.path.getsize(os.path.join(CONFIG['output_dir'], fn)) / 1024
    print(f'  {fn:40s} {sz:8.1f} KB')

## 9. Quick Inference Test (Ensemble)

In [ ]:
# Quick inference test using the ensemble model
from ml.model import load_model
from ml.transforms import get_val_transforms
from PIL import Image

ensemble_model = load_model(os.path.join(CONFIG['output_dir'], 'best_model.pth'), device)
ensemble_model.eval()
val_tfm = get_val_transforms()

# Pick a few samples to test
test_samples = full_dataset.samples[:5]

print(f'Inference test using ensemble ({CONFIG["n_folds"]} folds)')
print('=' * 65)
for s in test_samples:
    imgs = {
        'frontal': val_tfm(Image.open(s['frontal']).convert('RGB')).unsqueeze(0).to(device),
        'left_lateral': val_tfm(Image.open(s['left_lateral']).convert('RGB')).unsqueeze(0).to(device),
        'right_lateral': val_tfm(Image.open(s['right_lateral']).convert('RGB')).unsqueeze(0).to(device),
    }
    with torch.no_grad():
        results = ensemble_model.predict_scores(imgs['frontal'], imgs['left_lateral'], imgs['right_lateral'])
    
    pred_str = f'MGI={results["mgi"]["score"].item()} ({results["mgi"]["confidence"].item():.0f}%)  ' \
               f'OHI={results["ohi"]["score"].item()} ({results["ohi"]["confidence"].item():.0f}%)  ' \
               f'GEI={results["gei"]["score"].item()} ({results["gei"]["confidence"].item():.0f}%)'
    true_str = f'MGI={s["mgi"]}  OHI={s["ohi"]}  GEI={s["gei"]}'
    match = '\u2713' if (results['mgi']['score'].item() == s['mgi'] and 
                     results['ohi']['score'].item() == s['ohi'] and 
                     results['gei']['score'].item() == s['gei']) else '\u2717'
    print(f'Patient {s["sl_no"]:>3d}: True=[{true_str}]  Pred=[{pred_str}]  {match}')

In [ ]:
# Visual display of sample predictions
fig, axes = plt.subplots(len(test_samples), 3, figsize=(14, 4 * len(test_samples)))
for row, s in enumerate(test_samples):
    for col, (view, lbl) in enumerate([('frontal', 'Frontal'), ('left_lateral', 'Left Lat'), ('right_lateral', 'Right Lat')]):
        im = Image.open(s[view]).convert('RGB')
        ax = axes[row, col] if len(test_samples) > 1 else axes[col]
        ax.imshow(im)
        ax.set_title(f'P{s["sl_no"]} - {lbl}', fontsize=9)
        ax.axis('off')

    # Run prediction
    imgs = {v: val_tfm(Image.open(s[v]).convert('RGB')).unsqueeze(0).to(device)
            for v in ['frontal', 'left_lateral', 'right_lateral']}
    with torch.no_grad():
        results = ensemble_model.predict_scores(imgs['frontal'], imgs['left_lateral'], imgs['right_lateral'])
    
    ax = axes[row, 0] if len(test_samples) > 1 else axes[0]
    ax.set_ylabel(f'True: M{s["mgi"]} O{s["ohi"]} G{s["gei"]}\n'
                  f'Pred: M{results["mgi"]["score"].item()} O{results["ohi"]["score"].item()} G{results["gei"]["score"].item()}',
                  fontsize=8, rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

## Reference

| Component | Details |
|---|---|
| **Backbone** | DINOv2-base (ViT-B/14, 86M params, frozen) |
| **Classifier** | MLP (2304→512→256→3 heads), ~300K trainable params |
| **Training** | 5-fold stratified CV on cached features |
| **Loss** | Weighted CrossEntropy + label smoothing (0.1) |
| **Optimizer** | AdamW (lr=1e-3, wd=1e-3) + CosineAnnealing |
| **Inference** | Ensemble: average softmax from all 5 fold heads |
| **Data** | 201 samples, 3 views each (frontal, left lat, right lat) |
| **Targets** | MGI (0-4), OHI (0-3), GEI (0-2) |
| **GPU Optimizations** | AMP (float16), TF32, cuDNN benchmark, batch extraction |

### Output Files
- `fold_1_head.pth` .. `fold_5_head.pth` — Individual fold classifier heads
- `best_model.pth` — Copy of best fold (for single-model fallback)
- `ensemble_info.pth` — Ensemble metadata
- `training_history.json` — Full training history
- `dinov2_features.npz` — Cached DINOv2 feature vectors (reusable)
- `*.png` — Plots (confusion matrices, fold accuracies, training curves)